# DATA JUMLAH PENDUDUK #

* DATA

In [92]:
import pandas as pd

# ==============================================================================
# DATA
# ==============================================================================
df_penduduk = pd.read_csv('jumlah_penduduk.csv')

print("=========== DATA ===========")
df_penduduk

=========== DATA ===========


,no,kabupaten/kota,tahun,jumlah
0,1,KABUPATEN PACITAN,2018,554394
1,2,KABUPATEN PONOROGO,2018,870705
2,3,KABUPATEN TRENGGALEK,2018,694902
3,4,KABUPATEN TULUNGAGUNG,2018,1035290
4,5,KABUPATEN BLITAR,2018,1157500
...,...,...,...,...
261,262,KOTA PASURUAN,2024,219392
262,263,KOTA MOJOKERTO,2024,137393
263,264,KOTA MADIUN,2024,201767
264,265,KOTA SURABAYA,2024,2921996


* DATA CLEANING

In [93]:
#===============================================================================
# DATA CLEANING
#===============================================================================

print("=========== DATA CLEANING ===========")

# -------------------------
# 1. INFO DATA
# -------------------------
print("\n 1. Info Data:")
df_penduduk.info()

print("\n Info Data Setelah Konversi ke String (selain jumlah dan tahun)")
kolom_string = [
    'no'
]

df_penduduk[kolom_string] = df_penduduk[kolom_string].astype(str)

df_penduduk.info()

=========== DATA CLEANING ===========

 1. Info Data:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 266 entries, 0 to 265
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   no              266 non-null    int64 
 1   kabupaten/kota  266 non-null    object
 2   tahun           266 non-null    int64 
 3   jumlah          266 non-null    int64 
dtypes: int64(3), object(1)
memory usage: 8.4+ KB

 Info Data Setelah Konversi ke String (selain jumlah dan tahun)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 266 entries, 0 to 265
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   no              266 non-null    object
 1   kabupaten/kota  266 non-null    object
 2   tahun           266 non-null    int64 
 3   jumlah          266 non-null    int64 
dtypes: int64(2), object(2)
memory usage: 8.4+ KB


In [94]:
# -------------------------
# 2. STANDARDISASI KAB/KOT
# -------------------------
df_penduduk['kabupaten/kota'] = (
    df_penduduk['kabupaten/kota']
    .str.upper()
    .str.strip()
)

daftar_kabkot = [
    'KABUPATEN PACITAN',
    'KABUPATEN PONOROGO',
    'KABUPATEN TRENGGALEK',
    'KABUPATEN TULUNGAGUNG',
    'KABUPATEN BLITAR',
    'KABUPATEN KEDIRI',
    'KABUPATEN MALANG',
    'KABUPATEN LUMAJANG',
    'KABUPATEN JEMBER',
    'KABUPATEN BANYUWANGI',
    'KABUPATEN BONDOWOSO',
    'KABUPATEN SITUBONDO',
    'KABUPATEN PROBOLINGGO',
    'KABUPATEN PASURUAN',
    'KABUPATEN SIDOARJO',
    'KABUPATEN MOJOKERTO',
    'KABUPATEN JOMBANG',
    'KABUPATEN NGANJUK',
    'KABUPATEN MADIUN',
    'KABUPATEN MAGETAN',
    'KABUPATEN NGAWI',
    'KABUPATEN BOJONEGORO',
    'KABUPATEN TUBAN',
    'KABUPATEN LAMONGAN',
    'KABUPATEN GRESIK',
    'KABUPATEN BANGKALAN',
    'KABUPATEN SAMPANG',
    'KABUPATEN PAMEKASAN',
    'KABUPATEN SUMENEP',
    'KOTA KEDIRI',
    'KOTA BLITAR',
    'KOTA MALANG',
    'KOTA PROBOLINGGO',
    'KOTA PASURUAN',
    'KOTA MOJOKERTO',
    'KOTA MADIUN',
    'KOTA SURABAYA',
    'KOTA BATU'
]

# -------------------------
# 3. CEK KELENGKAPAN KAB/KOT
# -------------------------
tidak_ada = [
    nama for nama in daftar_kabkot
    if nama not in df_penduduk['kabupaten/kota'].values
]

print("\n2.Cek Kelengkapan Kabupaten/Kota")

if len(tidak_ada) > 0:
    print("Nama Kabupaten/kota yang tidak ada di dataset:")
    for nama in tidak_ada:
        print(nama)
else:
    print("Semua nama kabupaten/kota tersedia di dataset")


2.Cek Kelengkapan Kabupaten/Kota
Semua nama kabupaten/kota tersedia di dataset


In [95]:
# -------------------------
# 4. CEK DUPLIKAT
# -------------------------
jumlah_duplikat = df_penduduk.duplicated().sum()

print("\n3.Cek Data Duplikat")

if jumlah_duplikat > 0:
    print("Jumlah data duplikat:", jumlah_duplikat)
    df_penduduk = df_penduduk.drop_duplicates()
    print("Berhasil dihapus")
else:
    print("Tidak ada data duplikat")


3.Cek Data Duplikat
Tidak ada data duplikat


In [96]:
# -------------------------
# 5. CEK MISSING VALUE
# -------------------------
missing_value = df_penduduk.isnull().sum()

print("\n4.Cek Missing Value")

if missing_value.sum() > 0:
    print("Jumlah missing value:", missing_value.sum())

    kolom_kategori = 'kategori'
    kolom_numerik = df_penduduk.select_dtypes(include='number').columns

    for kolom in kolom_numerik:
        if df_penduduk[kolom].isna().sum() > 0:
            df_penduduk[kolom] = df_penduduk.groupby(kolom_kategori)[kolom].transform(
                lambda x: x.fillna(x.median())
            )

    print("Berhasil ditangani sesuai kategori")

else:
    print("Tidak ada missing value")


4.Cek Missing Value
Tidak ada missing value


In [97]:
# -------------------------
# 6. CEK OUTLIER (IQR)
# -------------------------
hasil_outlier = []
kolom_numerik = df_penduduk.select_dtypes(include='number').columns

print("\n5.Cek Outlier (IQR)")

for kolom in kolom_numerik:

    Q1 = df_penduduk[kolom].quantile(0.25)
    Q3 = df_penduduk[kolom].quantile(0.75)

    IQR = Q3 - Q1

    batas_bawah = Q1 - 1.5 * IQR
    batas_atas = Q3 + 1.5 * IQR

    jumlah_outlier = df_penduduk[
        (df_penduduk[kolom] < batas_bawah) |
        (df_penduduk[kolom] > batas_atas)
    ]

    if len(jumlah_outlier) > 0:
        keterangan = "Ada outlier"
        print(f"\nVariabel: {kolom}")
        print(f'Jumlah Outlier: {len(jumlah_outlier)}')
        display(jumlah_outlier[['kabupaten/kota', 'tahun', kolom]])
    else:
        print(f"\nVariabel: {kolom}")
        print("Tidak ada outlier")


5.Cek Outlier (IQR)

Variabel: tahun
Tidak ada outlier

Variabel: jumlah
Jumlah Outlier: 28


,kabupaten/kota,tahun,jumlah
6,KABUPATEN MALANG,2018,2591795
8,KABUPATEN JEMBER,2018,2440714
14,KABUPATEN SIDOARJO,2018,2216804
36,KOTA SURABAYA,2018,2885555
44,KABUPATEN MALANG,2019,2606204
46,KABUPATEN JEMBER,2019,2450668
52,KABUPATEN SIDOARJO,2019,2249476
74,KOTA SURABAYA,2019,2896195
82,KABUPATEN MALANG,2020,2619975
84,KABUPATEN JEMBER,2020,2459890


In [98]:
#-------------------------
# Rename kolom kabupaten/kota
#-------------------------
df_penduduk.rename(columns={'kabupaten/kota': 'nama_kabupaten_kota'}, inplace=True)
df_penduduk

,no,nama_kabupaten_kota,tahun,jumlah
0,1,KABUPATEN PACITAN,2018,554394
1,2,KABUPATEN PONOROGO,2018,870705
2,3,KABUPATEN TRENGGALEK,2018,694902
3,4,KABUPATEN TULUNGAGUNG,2018,1035290
4,5,KABUPATEN BLITAR,2018,1157500
...,...,...,...,...
261,262,KOTA PASURUAN,2024,219392
262,263,KOTA MOJOKERTO,2024,137393
263,264,KOTA MADIUN,2024,201767
264,265,KOTA SURABAYA,2024,2921996


* ISI VARIABEL IDENTITAS

In [99]:
import pandas as pd
# ==============================================================================
# ISI VARIABEL IDENTITAS
# ==============================================================================

print("=========== ISI VARIABEL IDENTITAS ===========")

# --------------------------
# 1. DATA MASTER VARIABEL IDENTITAS
# --------------------------

df_master = pd.read_csv('jumlah_penyakit_menular.csv')

df_master.columns = df_master.columns.str.strip()

# --------------------------
# 2. STANDARISASI NAMA KABUPATEN/KOTA 
# --------------------------
df_master['nama_kabupaten_kota'] = (
    df_master['nama_kabupaten_kota']
    .astype(str)
    .str.strip()
    .str.upper()
)

# --------------------------
# 3. MAPPING
# --------------------------
master_kabkot = (
    df_master[['kode_provinsi','nama_provinsi', 'kode_kabupaten_kota', 'nama_kabupaten_kota']]
    .drop_duplicates()
)

print("1. Data Master Variabel Identitas:") 
master_kabkot

=========== ISI VARIABEL IDENTITAS ===========
1. Data Master Variabel Identitas:


,kode_provinsi,nama_provinsi,kode_kabupaten_kota,nama_kabupaten_kota
0,35,JAWA TIMUR,3501,KABUPATEN PACITAN
4,35,JAWA TIMUR,3502,KABUPATEN PONOROGO
8,35,JAWA TIMUR,3503,KABUPATEN TRENGGALEK
12,35,JAWA TIMUR,3504,KABUPATEN TULUNGAGUNG
16,35,JAWA TIMUR,3505,KABUPATEN BLITAR
20,35,JAWA TIMUR,3506,KABUPATEN KEDIRI
24,35,JAWA TIMUR,3507,KABUPATEN MALANG
28,35,JAWA TIMUR,3508,KABUPATEN LUMAJANG
32,35,JAWA TIMUR,3509,KABUPATEN JEMBER
36,35,JAWA TIMUR,3510,KABUPATEN BANYUWANGI


In [100]:
# --------------------------
# 4. MERGE VARIABEL IDENTITAS
# --------------------------

# hapus kolom kode lama
df_penduduk = df_penduduk.drop(
    columns=['kode_provinsi', 'nama_provinsi', 'kode_kabupaten_kota'],
    errors='ignore'
)

# merge
df_penduduk = df_penduduk .merge(
    master_kabkot,
    on='nama_kabupaten_kota',
    how='left'
)

df_penduduk = df_penduduk.sort_values(
    ['tahun', 'kode_kabupaten_kota']
).reset_index(drop=True)

df_penduduk = df_penduduk[
    [
        'kode_provinsi',
        'nama_provinsi',
        'kode_kabupaten_kota',
        'nama_kabupaten_kota',
        'tahun',
        'jumlah'
        
    ]
]

df_penduduk_tahun_kab = df_penduduk.rename(columns={
    'jumlah': 'JUMLAH_PENDUDUK'
})

print("2. Data Merge Variabel Identitas")
df_penduduk_tahun_kab

2. Data Merge Variabel Identitas


,kode_provinsi,nama_provinsi,kode_kabupaten_kota,nama_kabupaten_kota,tahun,JUMLAH_PENDUDUK
0,35,JAWA TIMUR,3501,KABUPATEN PACITAN,2018,554394
1,35,JAWA TIMUR,3502,KABUPATEN PONOROGO,2018,870705
2,35,JAWA TIMUR,3503,KABUPATEN TRENGGALEK,2018,694902
3,35,JAWA TIMUR,3504,KABUPATEN TULUNGAGUNG,2018,1035290
4,35,JAWA TIMUR,3505,KABUPATEN BLITAR,2018,1157500
...,...,...,...,...,...,...
261,35,JAWA TIMUR,3575,KOTA PASURUAN,2024,219392
262,35,JAWA TIMUR,3576,KOTA MOJOKERTO,2024,137393
263,35,JAWA TIMUR,3577,KOTA MADIUN,2024,201767
264,35,JAWA TIMUR,3578,KOTA SURABAYA,2024,2921996


* SIMPAN DATA

In [101]:
#===============================================================================
# SIMPAN DATA
#===============================================================================
df_penduduk_tahun_kab.to_csv('data_jumlah_penduduk.csv', index=False)

In [102]:
print(df_penduduk_tahun_kab.columns.tolist())

['kode_provinsi', 'nama_provinsi', 'kode_kabupaten_kota', 'nama_kabupaten_kota', 'tahun', 'JUMLAH_PENDUDUK']
